In [11]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

con.execute(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT *
FROM read_parquet(
    '{REL}/fact_content_daily_performance/**/*.parquet'
)
""")

con.execute(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT *
FROM read_parquet(
    '{REL}/dim_content.parquet'
)
""")

print("DuckDB connection ready.")

HF token loaded: True
DuckDB connection ready.


In [2]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", bool(HF_TOKEN))

HF token loaded: True


# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lok997/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract

- **Unit of analysis:** One row represents one daily content-performance observation for a content item.
- **Tables used:** `dim_content` and `fact_daily`.
- **Time window:** March 2026 (`2026-03`), a mid-panel month.
- **Prediction/ranking target:** Predict whether a content item will show a future decline in performance.
- **Excluded:** Future/outcome-derived fields are excluded from the features because they would cause target leakage.

In [12]:
# Verify the grain:
# one row should represent one content item per day

con.sql("""
SELECT
    content_hash_id,
    report_date,
    COUNT(*) AS rows_per_content_day
FROM fact_daily
GROUP BY content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 10
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,rows_per_content_day
0,content_fc4ff20f189e0aa0,2026-06-16,2
1,content_ef8338f4365f7423,2026-06-20,2
2,content_b40e27e07768f907,2026-06-16,2
3,content_99a3e159bf3e5306,2026-06-16,2
4,content_023c4648db30b8c3,2026-06-17,2
5,content_19da6521d25c923a,2026-06-17,2
6,content_dacee0e0bac2dd31,2026-06-19,2
7,content_50331cbf0f4cf3f3,2026-06-22,2
8,content_cc5027aaab0fb4a0,2026-06-22,2
9,content_79441ebf6863362f,2026-06-22,2


In [13]:
# Verify March 2026 row count and date range

con.sql("""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM fact_daily
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
""").df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields

**Features**
- `gsc_impressions` — Search impressions available for the observed day.
- `gsc_clicks` — Search clicks available for the observed day.
- `gsc_avg_position` — Average search position available for the observed day.
- `ga4_pageviews` — Page views available for the observed day.
- `ga4_sessions` — Sessions available for the observed day.

**Label**
- `future_decline` — A derived label indicating whether the content item shows a future decline in performance.

**Context**
- `report_date` — Identifies the observation date.
- `client_hash_id` — Identifies the pseudonymized client.
- `content_hash_id` — Identifies the pseudonymized content item.
- `gsc_data_available` — Indicates whether GSC data is available for the observation.
- `ga4_data_available` — Indicates whether GA4 data is available for the observation.

**Excluded**
- Future/outcome-derived performance values — excluded because they would use information from the outcome period and cause target leakage.
- `client_hash_id` and `content_hash_id` are not used as predictive features because they are identifiers rather than meaningful performance signals.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
DESCRIBE fact_daily
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [16]:
con.sql("""
SELECT column_name
FROM (DESCRIBE fact_daily)
""").df()

,column_name
0,report_date
1,client_hash_id
2,content_hash_id
3,client_has_gsc
4,client_has_ga4
5,gsc_data_available
6,ga4_data_available
7,gsc_impressions
8,gsc_clicks
9,gsc_sum_position


In [9]:
# Connect DuckDB and create usable views for the warehouse tables

import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = 'hf://datasets/FlyRank/internship-warehouse'

con.execute(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT *
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
""")

con.execute(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT *
FROM read_parquet('{REL}/dim_content.parquet')
""")

print("DuckDB connected and views created successfully.")

DuckDB connected and views created successfully.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one —# Verify GSC availability for March 2026

con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM fact_daily
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
  AND gsc_data_available IS TRUE
""").df() typing sentences here breaks Run All.


SyntaxError: invalid syntax (1067545573.py, line 11)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.